# EGNN Physics-Informed Relaxation

Train an **Equivariant Graph Neural Network** to predict vertex displacements $\Delta X$ that
minimise the VertAX mechanical energy — replacing the bilevel optimisation after cell division.

## Architecture

**EGNN** (Satorras et al. 2021) adapted to the VertAX half-edge topology:

- Edge features: $\|\mathbf{x}_i - \mathbf{x}_j\|^2$, tension $\sigma_{ij}$ — **no absolute coordinates**
- Node feature update: MLP on aggregated messages
- Position update: $\mathbf{x}_i^{(l+1)} = \mathbf{x}_i^{(l)} + \sum_j \phi_x(m_{ij}) (\mathbf{x}_i^{(l)} - \mathbf{x}_j^{(l)})$
- Auto-regressive rollout at inference: apply one EGNN step 200 times

## Key invariances

1. **Translation invariance**: the network only sees $B X$ (relative displacements), never absolute positions
2. **Rotation equivariance**: position updates are sums of relative vectors $\mathbf{x}_i - \mathbf{x}_j$ weighted by scalars
3. **Batch masking**: block-diagonal incidence matrix $B$ prevents cross-tissue message passing

In [ ]:
import os
import sys
sys.path.insert(0, '/home/mcorbeau/Bureau/vertax/src/vertax/mitotic_spindle')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

import math
import numpy as np
import jax
import jax.numpy as jnp
from jax import vmap, jit, grad
import optax
import matplotlib.pyplot as plt
from pathlib import Path

from vertax import BoundedMesh
from vertax.topo import update_T1_bounded
from energy_adaptive import energy_adaptive

# ── Global constants ──────────────────────────────────────────────────────────
N_CELLS      = 20
N_MESHES     = 10000
MAX_EDGES    = 55
DATASET_PATH = Path('/home/mcorbeau/Bureau/vertax/src/vertax/mitotic_spindle/dataset_10k_20cells.pkl')

# ── Feature normalisation stats (computed on 20-cell meshes, random tensions) ─
# Update by running: FEAT_STATS = compute_feat_stats(dataset) in the dataset cell
FEAT_STATS = {
    'dist2_mean': 0.052, 'dist2_std': 0.058,
    'tens_mean' : 1.500, 'tens_std' : 0.289,
    'curv_mean' : 0.524, 'curv_std' : 0.256,
    'deg_mean'  : 5.476, 'deg_std'  : 0.895,
}

print('JAX devices:', jax.devices())

## 1. Incidence matrix and graph construction

From `heTable`, we extract the directed edge list `(src, tgt)` for each unique edge
and build the signed incidence matrix $B \in \mathbb{R}^{n_\text{edges} \times n_\text{verts}}$.

In [ ]:
DTYPE = jnp.float32
# MAX_EDGES, N_CELLS defined in imports cell

def build_graph(mesh: BoundedMesh):
    he      = np.array(mesh.edges)
    n_verts = mesh.nb_vertices

    he_even = he[::2]
    src_all = he_even[:, 3] + he_even[:, 5] - 2
    tgt_all = he_even[:, 4] + he_even[:, 6] - 3

    valid     = (src_all >= 0) & (tgt_all >= 0)
    srcs      = src_all[valid]
    tgts      = tgt_all[valid]
    valid_idx = np.where(valid)[0]
    n_edges   = len(srcs)

    assert n_edges <= MAX_EDGES, f'n_edges={n_edges} > MAX_EDGES={MAX_EDGES}'

    B = np.zeros((MAX_EDGES, n_verts), dtype=np.float32)
    for e, (s, t) in enumerate(zip(srcs, tgts)):
        B[e, s] = +1.0
        B[e, t] = -1.0

    he_params = np.array(mesh.edges_params)
    if he_params.size == 0:
        raw_tensions = np.full(n_edges, 1.5, dtype=np.float32)
    else:
        raw          = he_params[valid_idx]
        raw_tensions = np.array(jax.nn.sigmoid(jnp.array(raw)) + 1, dtype=np.float32)

    angles   = np.array(mesh.angles)
    raw_curv = angles[valid_idx].astype(np.float32)

    srcs_pad = np.zeros(MAX_EDGES, dtype=np.int32);   srcs_pad[:n_edges] = srcs
    tgts_pad = np.zeros(MAX_EDGES, dtype=np.int32);   tgts_pad[:n_edges] = tgts
    tens_pad = np.zeros(MAX_EDGES, dtype=np.float32); tens_pad[:n_edges] = raw_tensions
    curv_pad = np.zeros(MAX_EDGES, dtype=np.float32); curv_pad[:n_edges] = raw_curv

    arrays = dict(
        X    = np.array(mesh.vertices,        dtype=np.float32),
        he   = np.array(mesh.edges,           dtype=np.int32),
        ang  = np.array(mesh.angles,          dtype=np.float32),
        face = np.array(mesh.faces,           dtype=np.int32),
        vp   = np.array(mesh.vertices_params, dtype=np.float32),
        ep   = np.array(mesh.edges_params,    dtype=np.float32),
        fp   = np.array(mesh.faces_params,    dtype=np.float32),
    )

    return srcs_pad, tgts_pad, B.astype(np.float32), tens_pad, curv_pad, arrays


def build_batch(items: list):
    srcs_list = [it[0] for it in items]
    tgts_list = [it[1] for it in items]
    B_list    = [it[2] for it in items]
    tens_list = [it[3] for it in items]
    curv_list = [it[4] for it in items]
    arr_list  = [it[5] for it in items]

    n_verts_list = [a['X'].shape[0] for a in arr_list]
    total_verts  = sum(n_verts_list)
    total_edges  = MAX_EDGES * len(items)

    B_batch = np.zeros((total_edges, total_verts), dtype=np.float32)
    v_off, e_off = 0, 0
    vert_offsets = []
    for B_i, nv in zip(B_list, n_verts_list):
        vert_offsets.append(v_off)
        B_batch[e_off:e_off+MAX_EDGES, v_off:v_off+nv] = B_i
        v_off += nv
        e_off += MAX_EDGES

    srcs_b = np.concatenate([s + off for s, off in zip(srcs_list, vert_offsets)])
    tgts_b = np.concatenate([t + off for t, off in zip(tgts_list, vert_offsets)])
    tens_b = np.concatenate(tens_list)
    curv_b = np.concatenate(curv_list)

    stacked = {k: jnp.stack([jnp.array(a[k]) for a in arr_list]) for k in arr_list[0]}

    return (jnp.array(B_batch, dtype=DTYPE),
            jnp.array(srcs_b), jnp.array(tgts_b),
            jnp.array(tens_b, dtype=DTYPE), jnp.array(curv_b, dtype=DTYPE),
            vert_offsets, n_verts_list, stacked)


def init_mesh(mesh: BoundedMesh, target_area: float = 0.6, key=None) -> BoundedMesh:
    if key is None:
        key = jax.random.PRNGKey(0)
    mesh.vertices_params = jnp.zeros(mesh.nb_vertices)
    mesh.edges_params    = jax.random.uniform(key, (mesh.nb_edges,)) * 20 - 10
    mesh.faces_params    = jnp.full(mesh.nb_faces, target_area)
    return mesh


# Smoke test
mesh0 = BoundedMesh.from_random_seeds(nb_seeds=N_CELLS, width=math.sqrt(N_CELLS), height=math.sqrt(N_CELLS), random_key=0)
init_mesh(mesh0, key=jax.random.PRNGKey(0))
srcs, tgts, B, tensions, curvatures, arr0 = build_graph(mesh0)
assert B.shape == (MAX_EDGES, mesh0.nb_vertices), f'unexpected B shape: {B.shape}'
print(f'n_verts={arr0["X"].shape[0]}, B shape={B.shape} ✓')

mesh1 = BoundedMesh.from_random_seeds(nb_seeds=N_CELLS, width=math.sqrt(N_CELLS), height=math.sqrt(N_CELLS), random_key=1)
init_mesh(mesh1, key=jax.random.PRNGKey(1))
s1, t1, B1, ten1, curv1, arr1 = build_graph(mesh1)
B_b, srcs_b, tgts_b, tens_b, curv_b, v_offs, nv_list, stacked = build_batch([
    (srcs, tgts, B, tensions, curvatures, arr0),
    (s1,   t1,  B1, ten1,     curv1,     arr1),
])
print(f'Batch B={B_b.shape}, stacked X={stacked["X"].shape} ✓')

## 2. EGNN layers (JAX pure)

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
N_LAYERS   = 4
H_DIM      = 128
MSG_DIM    = 128

def mlp_init(key, layer_sizes, zero_last=False, small_last=False):
    params = []
    for i, (n_in, n_out) in enumerate(zip(layer_sizes[:-1], layer_sizes[1:])):
        key, sk = jax.random.split(key)
        is_last = (i == len(layer_sizes) - 2)
        if is_last and zero_last:
            W = jnp.zeros((n_in, n_out))
            b = jnp.zeros((n_out,))
        elif is_last and small_last:
            W = jax.random.normal(sk, (n_in, n_out)) * 1e-3
            b = jnp.zeros((n_out,))
        else:
            W = jax.random.normal(sk, (n_in, n_out)) * math.sqrt(2.0 / n_in)
            b = jnp.zeros((n_out,))
        params.append((W, b))
    return params


def mlp_apply(params, x, activate_last=False):
    for i, (W, b) in enumerate(params):
        x = x @ W + b
        if i < len(params) - 1 or activate_last:
            x = jax.nn.silu(x)
    return x


def layernorm_init(dim):
    return jnp.ones(dim), jnp.zeros(dim)


def layernorm_apply(params, x, eps=1e-6):
    gamma, beta = params
    mean = jnp.mean(x)
    var  = jnp.mean((x - mean) ** 2)
    return gamma * (x - mean) / jnp.sqrt(var + eps) + beta


def init_egnn(key, n_layers=N_LAYERS, h_dim=H_DIM, msg_dim=MSG_DIM):
    params = {}
    key, k = jax.random.split(key)
    params['embed'] = mlp_init(k, [1, h_dim, h_dim])

    params['layers'] = []
    for _ in range(n_layers):
        key, k1, k2, k3 = jax.random.split(key, 4)
        layer = {
            'phi_m': mlp_init(k1, [h_dim + h_dim + 3, msg_dim, msg_dim]),
            'phi_h': mlp_init(k2, [h_dim + msg_dim + 2, h_dim, h_dim]),
            'phi_x': mlp_init(k3, [msg_dim, h_dim, 1], small_last=True),
            'ln'   : layernorm_init(h_dim),
        }
        params['layers'].append(layer)

    return params


key = jax.random.PRNGKey(0)
params = init_egnn(key)
print('Parameters initialised.')
n_params = sum(
    W.size + b.size
    for p_list in [params['embed']] + [lp for l in params['layers'] for lp in l.values()]
    for W, b in (p_list if isinstance(p_list, list) else [p_list])
)
ln_params = sum(p.size for l in params['layers'] for p in l['ln'])
print(f'Total parameters: {n_params + ln_params:,}')

In [ ]:
def egnn_pass(params, X, B, srcs, tgts, tensions, curvatures, grad_X):
    degree = jnp.abs(B).sum(axis=0)

    deg_n = (degree - FEAT_STATS['deg_mean']) / FEAT_STATS['deg_std']
    H = vmap(lambda d: mlp_apply(params['embed'], d[None]))(deg_n)

    for layer in params['layers']:
        dX    = B @ X
        dist2 = (jnp.sum(dX**2, axis=1, keepdims=True) - FEAT_STATS['dist2_mean']) / FEAT_STATS['dist2_std']
        sigma = (tensions[:, None] - FEAT_STATS['tens_mean']) / FEAT_STATS['tens_std']
        kappa = (curvatures[:, None] - FEAT_STATS['curv_mean']) / FEAT_STATS['curv_std']

        h_src  = H[srcs]
        h_tgt  = H[tgts]
        msg_in = jnp.concatenate([h_src, h_tgt, dist2, sigma, kappa], axis=1)
        M      = vmap(lambda x: mlp_apply(layer['phi_m'], x))(msg_in)

        agg  = jnp.abs(B).T @ M
        h_in = jnp.concatenate([H, agg, grad_X], axis=1)
        H    = vmap(lambda x: mlp_apply(layer['phi_h'], x))(h_in)
        H    = vmap(lambda h: layernorm_apply(layer['ln'], h))(H)

        w        = jnp.tanh(vmap(lambda m: mlp_apply(layer['phi_x'], m))(M)) * 0.3
        X_update = B.T @ (w * (B @ X))
        X        = X + X_update / degree[:, None].clip(1.0)

    return X


def egnn_forward(params, X, B, srcs, tgts, tensions, curvatures,
                 he, ang, face, vp, ep, fp):
    """One EGNN step: compute ∂E/∂X, then apply egnn_pass once."""
    grad_X = jax.grad(lambda Xg: energy_adaptive(
        Xg, ang, he, face, None, None, None, vp, ep, fp
    ))(X)
    grad_norm = jnp.linalg.norm(grad_X) + 1e-8
    grad_X_n  = grad_X / grad_norm
    return egnn_pass(params, X, B, srcs, tgts, tensions, curvatures, grad_X_n)


def egnn_rollout(params, X, B, srcs, tgts, tensions, curvatures,
                 he, ang, face, vp, ep, fp, n_steps=200):
    """Apply egnn_forward iteratively for n_steps."""
    for _ in range(n_steps):
        X = egnn_forward(params, X, B, srcs, tgts, tensions, curvatures,
                         he, ang, face, vp, ep, fp)
    return X


# Smoke test
X0     = jnp.array(arr0['X'])
X_pred = egnn_forward(params, X0, jnp.array(B), jnp.array(srcs), jnp.array(tgts),
                      jnp.array(tensions), jnp.array(curvatures),
                      jnp.array(arr0['he']), jnp.array(arr0['ang']),
                      jnp.array(arr0['face']), jnp.array(arr0['vp']),
                      jnp.array(arr0['ep']), jnp.array(arr0['fp']))
print(f'Single step |ΔX| max: {jnp.max(jnp.abs(X_pred - X0)):.2e}')

## 3. Auto-regressive MSE loss

The network learns to imitate one gradient descent step at a time.
At inference, `egnn_rollout` applies the step 200 times.

In [ ]:
# ── Gradient descent target ───────────────────────────────────────────────────
GD_ALPHA  = 1e-3
GD_STEPS  = 200  # auto-regressive steps — matches rollout at inference


def compute_gd_target(X, he, ang, face, vp, ep, fp):
    g = jax.grad(lambda Xg: energy_adaptive(
        Xg, ang, he, face, None, None, None, vp, ep, fp
    ))(X)
    return X - GD_ALPHA * g


def physics_loss_single(params, X, B, srcs, tgts, tensions, curvatures,
                        he, ang, face, vp, ep, fp):
    def step_fn(_, carry):
        X_cur, total_loss = carry
        X_target = compute_gd_target(X_cur, he, ang, face, vp, ep, fp)
        X_pred   = egnn_forward(params, X_cur, B, srcs, tgts, tensions, curvatures,
                                he, ang, face, vp, ep, fp)
        loss_step = jnp.mean((X_pred - X_target) ** 2)
        return jax.lax.stop_gradient(X_target), total_loss + loss_step

    _, total_loss = jax.lax.fori_loop(
        0, GD_STEPS, step_fn, (X, jnp.zeros(()))
    )
    return total_loss / GD_STEPS


def physics_loss_batch(params, B_batch, srcs_b, tgts_b, tens_b, curv_b, stacked):
    batch_size = stacked['X'].shape[0]
    nv         = stacked['X'].shape[1]
    ne         = MAX_EDGES

    def mesh_loss(i):
        X_i    = stacked['X'][i]
        B_i    = jax.lax.dynamic_slice(B_batch, (i*ne, i*nv), (ne, nv))
        srcs_i = jax.lax.dynamic_slice(srcs_b,  (i*ne,),      (ne,)) - i*nv
        tgts_i = jax.lax.dynamic_slice(tgts_b,  (i*ne,),      (ne,)) - i*nv
        tens_i = jax.lax.dynamic_slice(tens_b,  (i*ne,),      (ne,))
        curv_i = jax.lax.dynamic_slice(curv_b,  (i*ne,),      (ne,))
        he_i   = stacked['he'][i];   ang_i  = stacked['ang'][i]
        face_i = stacked['face'][i]; vp_i   = stacked['vp'][i]
        ep_i   = stacked['ep'][i];   fp_i   = stacked['fp'][i]
        return physics_loss_single(params, X_i, B_i, srcs_i, tgts_i, tens_i, curv_i,
                                   he_i, ang_i, face_i, vp_i, ep_i, fp_i)

    return jnp.mean(vmap(mesh_loss)(jnp.arange(batch_size)))


@jit
def train_step(params, opt_state, B_batch, srcs_b, tgts_b, tens_b, curv_b, stacked):
    loss, grads = jax.value_and_grad(physics_loss_batch)(
        params, B_batch, srcs_b, tgts_b, tens_b, curv_b, stacked
    )
    updates, new_opt_state = optimizer.update(grads, opt_state, params)
    new_params = optax.apply_updates(params, updates)
    return new_params, new_opt_state, loss


# Smoke test
X0    = jnp.array(arr0['X'])
B0    = jnp.array(B)
he0   = jnp.array(arr0['he']);  ang0  = jnp.array(arr0['ang'])
face0 = jnp.array(arr0['face']); vp0  = jnp.array(arr0['vp'])
ep0   = jnp.array(arr0['ep']);  fp0   = jnp.array(arr0['fp'])

X_target = compute_gd_target(X0, he0, ang0, face0, vp0, ep0, fp0)
E_init   = float(energy_adaptive(X0,       ang0, he0, face0, None, None, None, vp0, ep0, fp0))
E_target = float(energy_adaptive(X_target, ang0, he0, face0, None, None, None, vp0, ep0, fp0))
print(f'E_init={E_init:.2f}  E_target(1 step)={E_target:.2f}  ΔE={E_init-E_target:.2f}')

loss_val = physics_loss_single(
    params, X0, B0, jnp.array(srcs), jnp.array(tgts),
    jnp.array(tensions), jnp.array(curvatures),
    he0, ang0, face0, vp0, ep0, fp0,
)
print(f'Single mesh loss (GD_STEPS={GD_STEPS}): {loss_val:.6f}')
print('Smoke test OK.')

## 4. Overfit test on a single mesh

In [ ]:
import optax

mesh_train = BoundedMesh.from_random_seeds(nb_seeds=N_CELLS, width=math.sqrt(N_CELLS), height=math.sqrt(N_CELLS), random_key=1)
init_mesh(mesh_train, key=jax.random.PRNGKey(1))
srcs_t, tgts_t, B_t, tensions_t, curvatures_t, arr_t = build_graph(mesh_train)

N_OVERFIT       = 500
LR_OVERFIT      = 3e-4
WEIGHT_DECAY_OV = 1e-4

schedule_ov = optax.cosine_decay_schedule(
    init_value=LR_OVERFIT, decay_steps=N_OVERFIT, alpha=1e-6/LR_OVERFIT
)
optimizer_ov = optax.chain(
    optax.clip_by_global_norm(0.5),
    optax.adamw(learning_rate=schedule_ov, weight_decay=WEIGHT_DECAY_OV),
)
key = jax.random.PRNGKey(42)
params_ov = init_egnn(key)
opt_state_ov = optimizer_ov.init(params_ov)

_loss_and_grad_ov = jit(jax.value_and_grad(
    lambda p: physics_loss_single(
        p,
        jnp.array(arr_t['X']), jnp.array(B_t),
        jnp.array(srcs_t), jnp.array(tgts_t),
        jnp.array(tensions_t), jnp.array(curvatures_t),
        jnp.array(arr_t['he']), jnp.array(arr_t['ang']), jnp.array(arr_t['face']),
        jnp.array(arr_t['vp']), jnp.array(arr_t['ep']), jnp.array(arr_t['fp']),
    )
))

losses_ov = []
for step in range(N_OVERFIT+1):
    loss, grads = _loss_and_grad_ov(params_ov)
    updates, opt_state_ov = optimizer_ov.update(grads, opt_state_ov, params_ov)
    params_ov = optax.apply_updates(params_ov, updates)
    losses_ov.append(float(loss))
    if step % 100 == 0:
        print(f'Step {step:4d} | loss = {loss:.6f}')

plt.figure(figsize=(8, 3))
plt.plot(losses_ov)
plt.xlabel('Step'); plt.ylabel('MSE loss'); plt.title('Overfit test — single mesh')
plt.yscale('log'); plt.tight_layout(); plt.show()

## 5. Training on a dataset of random meshes

In [ ]:
import pickle

# N_MESHES, N_CELLS, DATASET_PATH, MAX_EDGES defined in imports cell

def compute_feat_stats(dataset, n_sample=500):
    sample = dataset[:n_sample]
    valid_counts = [int(np.sum(item[0] > 0)) for item in sample]
    all_tens = np.concatenate([item[3][:vc] for item, vc in zip(sample, valid_counts)])
    all_curv = np.concatenate([item[4][:vc] for item, vc in zip(sample, valid_counts)])
    all_dist2, all_deg = [], []
    for item in sample:
        X  = jnp.array(item[5]['X'])
        Bm = jnp.array(item[2])
        dX = Bm @ X
        all_dist2.append(np.array(jnp.sum(dX**2, axis=1)))
        all_deg.append(np.array(jnp.abs(Bm).sum(axis=0)))
    all_dist2 = np.concatenate(all_dist2)
    all_deg   = np.concatenate(all_deg)
    return {
        'dist2_mean': float(np.mean(all_dist2)), 'dist2_std': float(np.std(all_dist2) + 1e-8),
        'tens_mean' : float(np.mean(all_tens)),  'tens_std' : float(np.std(all_tens)  + 1e-8),
        'curv_mean' : float(np.mean(all_curv)),  'curv_std' : float(np.std(all_curv)  + 1e-8),
        'deg_mean'  : float(np.mean(all_deg)),   'deg_std'  : float(np.std(all_deg)   + 1e-8),
    }

# ── Generate (run once, then comment out) ────────────────────────────────────
# from joblib import Parallel, delayed
# def generate_one(i):
#     m = BoundedMesh.from_random_seeds(nb_seeds=N_CELLS,
#                                       width=math.sqrt(N_CELLS),
#                                       height=math.sqrt(N_CELLS),
#                                       random_key=i)
#     init_mesh(m, key=jax.random.PRNGKey(i))
#     s, t, Bm, tens, curv, arrays = build_graph(m)
#     return (s, t, Bm, tens, curv, arrays)
# print(f'Generating {N_MESHES} meshes...')
# dataset = Parallel(n_jobs=-1)(delayed(generate_one)(i) for i in range(N_MESHES))
# with open(DATASET_PATH, 'wb') as f:
#     pickle.dump(dataset, f)
# print(f'Saved to {DATASET_PATH}')

# ── Load ──────────────────────────────────────────────────────────────────────
print(f'Loading dataset from {DATASET_PATH}...')
with open(DATASET_PATH, 'rb') as f:
    raw = pickle.load(f)

needs_convert = False
if hasattr(raw[0][0], 'nb_vertices'):
    needs_convert = True
elif raw[0][2].shape[0] != MAX_EDGES:
    needs_convert = True

if needs_convert:
    print(f'Converting to padded format (MAX_EDGES={MAX_EDGES})...')
    dataset = []
    for item in raw:
        if hasattr(item[0], 'nb_vertices'):
            m = item[0]
        else:
            raise ValueError('Cannot convert — re-generate the dataset.')
        s, t, Bm, tens, curv, arrays = build_graph(m)
        dataset.append((s, t, Bm, tens, curv, arrays))
        if len(dataset) % 1000 == 0:
            print(f'  {len(dataset)}/{N_MESHES}')
    with open(DATASET_PATH, 'wb') as f:
        pickle.dump(dataset, f)
    print('Converted and saved.')
else:
    dataset = raw

print(f'Loaded {len(dataset)} meshes. B shape: {dataset[0][2].shape}')

FEAT_STATS = compute_feat_stats(dataset)
print('Feature stats:')
for k, v in FEAT_STATS.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
import wandb
import pickle

BATCH_SIZE   = 128
LR_PEAK      = 1e-3
LR_MIN       = 1e-6
WARMUP       = 200
N_EPOCHS     = 20000
LOG_EVERY    = 50
GRAD_CLIP    = 0.5
WEIGHT_DECAY = 1e-4
MODEL_PATH   = DATASET_PATH.parent / 'egnn_params.pkl'

schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0, peak_value=LR_PEAK,
    warmup_steps=WARMUP, decay_steps=N_EPOCHS, end_value=LR_MIN,
)
optimizer = optax.chain(
    optax.clip_by_global_norm(GRAD_CLIP),
    optax.adamw(learning_rate=schedule, weight_decay=WEIGHT_DECAY),
)

key = jax.random.PRNGKey(99)
params = init_egnn(key)
opt_state = optimizer.init(params)

losses = []
rng = np.random.default_rng(0)

# ── Reference mesh for diagnostic ─────────────────────────────────────────────
_ref = dataset[0]
_s_r, _t_r, _B_r, _te_r, _cu_r, _arr_r = _ref
_X_r    = jnp.array(_arr_r['X'])
_B_r    = jnp.array(_B_r)
_s_r    = jnp.array(_s_r)
_t_r    = jnp.array(_t_r)
_te_r   = jnp.array(_te_r)
_cu_r   = jnp.array(_cu_r)
_ang_r  = jnp.array(_arr_r['ang']);  _he_r   = jnp.array(_arr_r['he'])
_face_r = jnp.array(_arr_r['face']); _vp_r   = jnp.array(_arr_r['vp'])
_ep_r   = jnp.array(_arr_r['ep']);   _fp_r   = jnp.array(_arr_r['fp'])
_E_init = float(energy_adaptive(_X_r, _ang_r, _he_r, _face_r, None, None, None, _vp_r, _ep_r, _fp_r))
print(f'Reference: E_init={_E_init:.2f}')

wandb.init(
    project='egnn-relaxation',
    config=dict(
        n_layers=N_LAYERS, h_dim=H_DIM, msg_dim=MSG_DIM,
        n_meshes=N_MESHES, n_cells=N_CELLS, max_edges=MAX_EDGES,
        n_epochs=N_EPOCHS, batch_size=BATCH_SIZE,
        lr_peak=LR_PEAK, lr_min=LR_MIN, warmup=WARMUP,
        features='dist2+tension+curvature+grad_X', layernorm=True, normalised=True,
        schedule='warmup_cosine_decay', grad_clip=GRAD_CLIP, tanh_clamp=0.3,
        loss='MSE_autoregressive', gd_alpha=GD_ALPHA, gd_steps=GD_STEPS,
        optimizer='adamw', weight_decay=WEIGHT_DECAY,
    ),
)

for epoch in range(N_EPOCHS):
    idx   = rng.integers(N_MESHES, size=BATCH_SIZE)
    items = [dataset[i] for i in idx]
    B_b, srcs_b, tgts_b, tens_b, curv_b, _, _, stacked = build_batch(items)

    params, opt_state, loss = train_step(
        params, opt_state, B_b, srcs_b, tgts_b, tens_b, curv_b, stacked
    )
    loss_val = float(loss)

    if np.isnan(loss_val) or np.isinf(loss_val):
        print(f'NaN/Inf at epoch {epoch+1} — stopping.')
        wandb.log({'loss': float('nan')}, step=epoch)
        break

    losses.append(loss_val)

    if (epoch + 1) % LOG_EVERY == 0:
        mean_loss = np.mean(losses[-LOG_EVERY:])
        lr_current = float(schedule(epoch))
        wandb.log({'loss': mean_loss, 'lr': lr_current}, step=epoch)
        if (epoch + 1) % 1000 == 0:
            X_roll_r = egnn_rollout(params, _X_r, _B_r, _s_r, _t_r, _te_r, _cu_r,
                                    _he_r, _ang_r, _face_r, _vp_r, _ep_r, _fp_r, n_steps=200)
            E_roll_r = float(energy_adaptive(X_roll_r, _ang_r, _he_r, _face_r, None, None, None, _vp_r, _ep_r, _fp_r))
            wandb.log({'E_rollout_200': E_roll_r}, step=epoch)
            print(f'Epoch {epoch+1:6d} | mean loss = {mean_loss:.6f} | lr = {lr_current:.2e} | E_rollout200={E_roll_r:.2f}')

# ── Save model ────────────────────────────────────────────────────────────────
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(params, f)
print(f'Model saved to {MODEL_PATH}')

wandb.finish()

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('MSE loss')
plt.title(f'Training — autoregressive GD_STEPS={GD_STEPS}, batch={BATCH_SIZE}')
plt.yscale('log'); plt.tight_layout(); plt.show()

## 6. Evaluation: post-division relaxation

Compare EGNN rollout (200 steps) vs VertAX bilevel (2000 iterations).

In [ ]:
# ── Plot 3 random meshes: original vs EGNN rollout ───────────────────────────
from vertax import get_plot_mesh, EdgePlot, FacePlot, VertexPlot

PLOT_KW = dict(
    edge_plot=EdgePlot.EDGE_PARAMETER, edge_parameters_name='tension',
    face_plot=FacePlot.AREA, face_parameters_name='area',
    vertex_plot=VertexPlot.INVISIBLE,
)

rng_plot = np.random.default_rng(42)
plot_indices = rng_plot.integers(N_MESHES, size=3).tolist()

fig, axes = plt.subplots(3, 2, figsize=(12, 18))

for row, idx in enumerate(plot_indices):
    item = dataset[idx]
    s_i, t_i, B_i, te_i, cu_i, arr_i = item

    X_i    = jnp.array(arr_i['X'])
    X_roll = egnn_rollout(params, X_i, jnp.array(B_i),
                          jnp.array(s_i), jnp.array(t_i), jnp.array(te_i), jnp.array(cu_i),
                          jnp.array(arr_i['he']), jnp.array(arr_i['ang']),
                          jnp.array(arr_i['face']), jnp.array(arr_i['vp']),
                          jnp.array(arr_i['ep']), jnp.array(arr_i['fp']), n_steps=200)

    E_init_i = float(energy_adaptive(X_i,    jnp.array(arr_i['ang']), jnp.array(arr_i['he']),
                                     jnp.array(arr_i['face']), None, None, None,
                                     jnp.array(arr_i['vp']), jnp.array(arr_i['ep']), jnp.array(arr_i['fp'])))
    E_roll_i = float(energy_adaptive(X_roll,  jnp.array(arr_i['ang']), jnp.array(arr_i['he']),
                                     jnp.array(arr_i['face']), None, None, None,
                                     jnp.array(arr_i['vp']), jnp.array(arr_i['ep']), jnp.array(arr_i['fp'])))

    m_orig = BoundedMesh.from_random_seeds(nb_seeds=N_CELLS,
                                           width=math.sqrt(N_CELLS),
                                           height=math.sqrt(N_CELLS),
                                           random_key=idx)
    init_mesh(m_orig, key=jax.random.PRNGKey(idx))
    m_roll = BoundedMesh.copy_mesh(m_orig)
    m_roll.vertices = X_roll

    for col, (m, title) in enumerate([
        (m_orig, f'Original (E={E_init_i:.1f})'),
        (m_roll, f'EGNN rollout 200 (E={E_roll_i:.1f})'),
    ]):
        mc = BoundedMesh.copy_mesh(m)
        mc.angles = jnp.clip(mc.angles, 0.017, jnp.pi / 2 - 0.01)
        fig_m, _ = get_plot_mesh(mc, title=title, **PLOT_KW)
        fig_m.canvas.draw()
        w, h = fig_m.canvas.get_width_height()
        img = np.frombuffer(fig_m.canvas.tostring_argb(), dtype=np.uint8).reshape(h, w, 4)
        img = np.roll(img, -1, axis=2)
        plt.close(fig_m)
        axes[row, col].imshow(img)
        axes[row, col].set_title(title, fontsize=11)
        axes[row, col].axis('off')

plt.suptitle('3 random meshes: original vs EGNN rollout 200 steps', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
import importlib
import boundedbop_patched; importlib.reload(boundedbop_patched)
import energy_adaptive as ea_mod; importlib.reload(ea_mod)
import cell_division as cd_mod; importlib.reload(cd_mod)
import spindle_model as sm_mod; importlib.reload(sm_mod)
from boundedbop_patched import BoundedBilevelOptimizer
from vertax import get_plot_mesh, EdgePlot, FacePlot, VertexPlot
from cell_division import divide_cell
from spindle_model import find_spindle

PLOT_KW = dict(
    edge_plot=EdgePlot.EDGE_PARAMETER, edge_parameters_name='tension',
    face_plot=FacePlot.AREA, face_parameters_name='area',
    vertex_plot=VertexPlot.INVISIBLE,
)

def _clip(m):
    mc = BoundedMesh.copy_mesh(m)
    mc.angles = jnp.clip(mc.angles, 0.017, jnp.pi / 2 - 0.01)
    return mc

def _energy(m):
    return float(ea_mod.energy_adaptive(
        jnp.array(m.vertices), jnp.array(m.angles), jnp.array(m.edges),
        jnp.array(m.faces), None, None, None,
        jnp.array(m.vertices_params), jnp.array(m.edges_params), jnp.array(m.faces_params),
    ))

def _render(m, title):
    fig_m, _ = get_plot_mesh(_clip(m), title=title, **PLOT_KW)
    fig_m.canvas.draw()
    w, h = fig_m.canvas.get_width_height()
    img = np.frombuffer(fig_m.canvas.tostring_argb(), dtype=np.uint8).reshape(h, w, 4)
    img = np.roll(img, -1, axis=2)
    plt.close(fig_m)
    return img

# ── 1. Build and relax initial mesh ──────────────────────────────────────────
mesh_base = BoundedMesh.from_random_seeds(nb_seeds=N_CELLS,
                                           width=math.sqrt(N_CELLS),
                                           height=math.sqrt(N_CELLS),
                                           random_key=N_MESHES + 1)
init_mesh(mesh_base, key=jax.random.PRNGKey(N_MESHES + 1))
opt_init = BoundedBilevelOptimizer()
opt_init.loss_function_inner = ea_mod.energy_adaptive
opt_init.inner_solver        = optax.adam(learning_rate=1e-4)
opt_init.update_T1           = True
opt_init._update_T1_func     = update_T1_bounded
opt_init.min_dist_T1         = 0.01
opt_init.max_nb_iterations   = 2000
opt_init.tolerance           = 1e-6
opt_init.patience            = 50
opt_init.inner_optimization(mesh_base)
print(f'Initial mesh relaxed — {mesh_base.nb_faces} cells, E = {_energy(mesh_base):.4f}')

# ── 2. Divide one cell ────────────────────────────────────────────────────────
rng_pick = np.random.default_rng(7)
face_id = c = u = None
for fid in rng_pick.permutation(mesh_base.nb_faces).tolist():
    try:
        c_try, u_try, _, _ = find_spindle(fid, mesh_base, d_fraction=0.3, n_gauss=12)
        mesh_test = BoundedMesh.copy_mesh(mesh_base)
        divide_cell(mesh_test, fid, c_try, u_try)
        face_id, c, u = fid, c_try, u_try
        break
    except Exception:
        continue

assert face_id is not None
mesh_post, _ = divide_cell(mesh_base, face_id, c, u)
total_area   = float(jnp.sum(mesh_base.get_area(jnp.arange(mesh_base.nb_faces))))
mesh_post.faces_params = jnp.full(mesh_post.nb_faces, total_area / mesh_post.nb_faces)
print(f'Post-division — {mesh_post.nb_faces} cells, E = {_energy(mesh_post):.4f}')

# ── 3. Bilevel reference ──────────────────────────────────────────────────────
mesh_bilevel = BoundedMesh.copy_mesh(mesh_post)
opt_ref = BoundedBilevelOptimizer()
opt_ref.loss_function_inner = ea_mod.energy_adaptive
opt_ref.inner_solver        = optax.adam(learning_rate=1e-4)
opt_ref.update_T1           = True
opt_ref._update_T1_func     = update_T1_bounded
opt_ref.min_dist_T1         = 0.01
opt_ref.max_nb_iterations   = 2000
opt_ref.tolerance           = 1e-6
opt_ref.patience            = 50
opt_ref.inner_optimization(mesh_bilevel)
E_bilevel = _energy(mesh_bilevel)
print(f'Bilevel relaxed  — E = {E_bilevel:.4f}')

# ── 4. EGNN rollout (200 steps) ───────────────────────────────────────────────
init_mesh(mesh_post, key=jax.random.PRNGKey(42))
srcs_p, tgts_p, B_p, tens_p, curv_p, arr_p = build_graph(mesh_post)
X_post  = jnp.array(arr_p['X'])
X_egnn  = egnn_rollout(params, X_post, jnp.array(B_p),
                       jnp.array(srcs_p), jnp.array(tgts_p),
                       jnp.array(tens_p), jnp.array(curv_p),
                       jnp.array(arr_p['he']), jnp.array(arr_p['ang']),
                       jnp.array(arr_p['face']), jnp.array(arr_p['vp']),
                       jnp.array(arr_p['ep']), jnp.array(arr_p['fp']),
                       n_steps=200)
mesh_egnn = BoundedMesh.copy_mesh(mesh_post)
mesh_egnn.vertices = X_egnn
E_egnn = _energy(mesh_egnn)
print(f'EGNN rollout 200 steps — E = {E_egnn:.4f}')

# ── 5. Plot ───────────────────────────────────────────────────────────────────
E_post = _energy(mesh_post)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (m, title) in zip(axes, [
    (mesh_post,    f'Post-division\n(E = {E_post:.2f})'),
    (mesh_egnn,    f'EGNN 200 steps\n(E = {E_egnn:.2f})'),
    (mesh_bilevel, f'Bilevel 2000 iters\n(E = {E_bilevel:.2f})'),
]):
    ax.imshow(_render(m, title))
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('Post-division relaxation: EGNN rollout vs VertAX bilevel', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()